# Cross-validation and tuning

The impact estimator is validated with an expanding-window time-series cross-validation: five folds, each fitted on every month before its validation month (May to September 2025). Rows are never shuffled, because emergencies are ordered in time and follow the seasons, and the October–December test months are never used here. The same folds tune the regularisation strength C of the logistic regression and the learning rate and depth of gradient boosting, and compare them with the type-and-location baseline. PR-AUC is used because high-impact cases are about 16% of the records.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ml.data import load_frame
from ml.evaluation import (
    best_f1_threshold,
    classification_metrics,
    rolling_origin_folds,
    split_dates,
    temporal_masks,
)
from ml.features import report_time_features
from ml.impact import (
    GRADIENT_BOOSTING,
    LOGISTIC_REGRESSION,
    TIE_MARGIN,
    TypeLocationBaseline,
    cross_validate_impact,
    fit_pipeline,
)
from ml.tuning import IMPACT_GRID, tune_impact

frame = load_frame(DATA / "ungrd_emergencies_v2.csv", DATA / "divipola_dane.csv")
labels = frame["is_high_impact"].to_numpy(dtype=bool)
split = split_dates(frame["occurred_on"])

## Folds

`rolling_origin_folds` returns one fit mask and one validation mask per fold. Each fold is fitted on all the months before its validation month, so the training window grows by one month at a time.

In [ ]:
folds = rolling_origin_folds(frame["occurred_on"], split.test_start)
dates = pd.to_datetime(frame["occurred_on"])
pd.DataFrame(
    [
        {
            "fold": number,
            "fit months": f"{dates[fold['fit']].min():%b} to {dates[fold['fit']].max():%b}",
            "validation month": f"{dates[fold['validation']].min():%B}",
            "fit rows": int(fold["fit"].sum()),
            "validation rows": int(fold["validation"].sum()),
            "validation high impact": int(labels[fold["validation"].to_numpy()].sum()),
        }
        for number, fold in enumerate(folds, start=1)
    ]
).set_index("fold")

## Hyperparameter search

`tune_impact` scores every configuration of the grid on the five folds and reports the mean and the standard deviation of PR-AUC, best first. The current defaults are marked.

In [ ]:
search = tune_impact(frame)
search.round(3)

## Spread across folds

PR-AUC of each fold for every configuration and for the type-and-location baseline. Points are the folds; bars are their mean. A wide spread means the score depends on the month.

In [ ]:
def label(kind, params):
    short = "LR" if kind == LOGISTIC_REGRESSION else "GB"
    return f"{short} " + ", ".join(f"{name}={value}" for name, value in params.items())


fold_scores = {
    label(kind, params): cross_validate_impact(frame, kind, params)["fold_pr_auc"]
    for kind, grid in IMPACT_GRID.items()
    for params in grid
}
fold_scores["Baseline"] = [
    classification_metrics(
        labels[fold["validation"].to_numpy()],
        TypeLocationBaseline().fit(frame[fold["fit"]]).score(frame[fold["validation"]]),
        0.5,
    )["pr_auc"]
    for fold in folds
]

names = list(fold_scores)
means = [np.mean(fold_scores[name]) for name in names]
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(names)), means, color="#9bbfb9")
for position, name in enumerate(names):
    ax.scatter([position] * len(fold_scores[name]), fold_scores[name], color="#032125", s=12)
ax.set_xticks(range(len(names)), names, rotation=45, ha="right")
ax.set_ylabel("PR-AUC")
ax.set_title("PR-AUC by fold (May to September)")
plt.tight_layout()
plt.show()

## Selected configuration and the test period

A configuration replaces the default of its kind only if its mean PR-AUC is higher by more than `TIE_MARGIN` (0.02), the same margin used to choose between logistic regression and gradient boosting. The October–December test months were not used for any of these choices.

In [ ]:
rows = []
for kind in (LOGISTIC_REGRESSION, GRADIENT_BOOSTING):
    candidates = search[search["kind"] == kind]
    default, best = candidates[candidates["is_default"]].iloc[0], candidates.iloc[0]
    margin = best["pr_auc_mean"] - default["pr_auc_mean"]
    rows.append({
        "kind": kind,
        "default": default["params"],
        "default PR-AUC": default["pr_auc_mean"],
        "best": best["params"],
        "best PR-AUC": best["pr_auc_mean"],
        "margin": margin,
        "decision": "change the default" if margin > TIE_MARGIN else "keep the default",
    })
pd.DataFrame(rows).set_index("kind").round(3)

## Overfitting check

The selected logistic regression and gradient boosting are fitted on January–September with the threshold tuned on September, then scored on their own training rows and on the October–December test rows.

In [ ]:
masks = temporal_masks(frame, split_dates(frame["occurred_on"]))
fit, validation, train, test = (masks[k].to_numpy() for k in ("fit", "validation", "train", "test"))

results = {}
for kind in (LOGISTIC_REGRESSION, GRADIENT_BOOSTING):
    tuned = fit_pipeline(kind, frame[fit])
    threshold = best_f1_threshold(labels[validation], tuned.predict_proba(report_time_features(frame[validation]))[:, 1])
    model = fit_pipeline(kind, frame[train])
    for part, mask in (("train", train), ("test", test)):
        probability = model.predict_proba(report_time_features(frame[mask]))[:, 1]
        results[(kind, part)] = {"threshold": threshold, **classification_metrics(labels[mask], probability, threshold)}

pd.DataFrame(results).T.round(3)

## Recall by event type on the test set

Shows where the selected logistic regression misses high-impact emergencies.

In [ ]:
tuned = fit_pipeline(LOGISTIC_REGRESSION, frame[fit])
threshold = best_f1_threshold(labels[validation], tuned.predict_proba(report_time_features(frame[validation]))[:, 1])
model = fit_pipeline(LOGISTIC_REGRESSION, frame[train])
test_frame = frame[test].assign(
    flagged=model.predict_proba(report_time_features(frame[test]))[:, 1] >= threshold,
    actual=labels[test],
)
by_type = test_frame.groupby("event_type").apply(
    lambda group: pd.Series({
        "rows": len(group),
        "high_impact": int(group["actual"].sum()),
        "recall": (group["flagged"] & group["actual"]).sum() / max(group["actual"].sum(), 1),
        "share_flagged": group["flagged"].mean(),
    })
)
by_type.sort_values("rows", ascending=False).head(6).round(2)

## Target thresholds from training months only

The high-impact label uses per-type 90th percentiles of people affected and hectares, computed over the whole year. This check recomputes them with January–September only and retrains the impact estimator, to measure how much the test months influence the label.

In [ ]:
from ml.impact import train_impact


def percentile_90(data, column, reference):
    positive = reference[reference[column] > 0]
    return data["event_type"].map(positive.groupby("event_type")[column].quantile(0.9))


reference = frame[pd.to_datetime(frame["occurred_on"]) < pd.Timestamp(split_dates(frame["occurred_on"]).test_start)]
victims = frame["deceased"] + frame["injured"] + frame["missing"] > 0
people = (frame["people_affected"] > 0) & (frame["people_affected"] >= percentile_90(frame, "people_affected", reference))
hectares = (frame["hectares"] > 0) & (frame["hectares"] >= percentile_90(frame, "hectares", reference))
train_only = frame.assign(is_high_impact=victims | (frame["homes_destroyed"] > 0) | people | hectares)

print(f"Labels that change: {int((train_only['is_high_impact'] != frame['is_high_impact']).sum())} of {len(frame):,}")
keys = ["model", "f1", "recall", "precision", "pr_auc", "baseline_f1", "baseline_pr_auc"]
pd.DataFrame({
    "whole year": {k: train_impact(frame).metrics[k] for k in keys},
    "training months": {k: train_impact(train_only).metrics[k] for k in keys},
})